# Custom GluonTS Model

This notebook adapts the model outlined in [TEMPO](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://arxiv.org/pdf/2310.04948) to [GluonTS](https://ts.gluon.ai/stable/index.html) and follows GluonTS's [Custom Model Tutorial](https://ts.gluon.ai/stable/tutorials/advanced_topics/howto_pytorch_lightning.html). There are two main tasks we need to complete to adapt TEMPO to GluonTS:
1. ~~Create a [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) wrapper around TEMPO so we can perform training~~
2. Create a GluonTS [`Predictor`](https://ts.gluon.ai/stable/api/gluonts/gluonts.torch.model.predictor.html?highlight=predictor#module-gluonts.torch.model.predictor) so we can perform inference

## [Dummy Dataset](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Custom-datasets)

For now, we'll create a dummy dataset of randomly generated values to pass into the model. The dataset's values (i.e. the actual values of each time series) will be in a `numpy.array` and the dataset's indices (i.e. timestamps) will be in a `pandas.Period`.

**Once I finish adapting TEMPO's datasets to a GluonTS friendly form, I'll use those.**

First, we'll define the dataset's metadata and create its values and indices.

In [1]:
import numpy as np
import pandas as pd

# Number of time series to store in the dummy dataset
num_time_series = 256 * 2

# Number of time steps in each time series
num_time_steps = 336 * 2

# Define each time series's frequency
freq = "1H"

# Define the dataset's prediction length
prediction_length = 96  # hours

# Create randomy generated values to use in the dummy dataset
dummmy_values = np.random.normal(size=(num_time_series, num_time_steps))

# Create the starting timestamp of the dummy dataset
start = pd.Period("01-01-2019", freq=freq)

print(f"dummy_values.shape: {dummmy_values.shape}")
print(f"start: {start}")  # can be different for each dataset

dummy_values.shape: (512, 672)
start: 2019-01-01 00:00


Then, we'll split the dataset and bring it into a GluonTS appropriate form. And that's it! Our dataset is now ready to be used.

In [2]:
from gluonts.dataset.common import ListDataset

# Remove the prediction window from the data in the training set
train_data = dummmy_values[:, :-prediction_length]

# Create the training set
train_set = ListDataset(
    [{"target": time_series, "start": start} for time_series in train_data], freq=freq
)

# Create the test set and include the prediction window
test_set = ListDataset(
    [{"target": time_series, "start": start} for time_series in dummmy_values],
    freq=freq,
)

## Training

### PyTorch Lightning Wrapper

Before creating the PyTorch wrapper, we'll define the hyperparameters we need for training.

In [3]:
learning_rate = 1e-3
batch_size = 128
num_batches_per_epoch = 50
max_epochs = 1

# Number of past samples to use when computing forecasts
context_length = 2 * 7 * 24

Create a [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) wrapper to wrap around TEMPO so we can train the model using PyTorch Lightning.

In [4]:
import pytorch_lightning as pl
import torch
from gluonts.torch import PyTorchPredictor
from gluonts.torch.distributions import StudentTOutput
from gluonts.model.forecast_generator import DistributionForecastGenerator

from tempo.models.TEMPO import TEMPO


class LightningTEMPO(TEMPO, pl.LightningModule):
    def __init__(self, configs, args=None):
        super().__init__(configs)
        # TODO: change args default value so it's not None once you finish your prototype
        # Commmand line arguments
        self.args = args

        # Model configuration
        self.configs = configs

        # TODO: once you get a prototype working, change the code to allow for different output distributions
        # Type of distribution for model's output. We'll use a Student's t-distribution
        self.distr_output = StudentTOutput()

    def training_step(self, batch, batch_index):
        """
        Defines the logic for a single training loop iteration.
        """
        # Get past time series values
        past_target = batch["past_target"]

        # Get future time series values
        future_target = batch["future_target"]

        # TODO: figure out how to get trend, seasonal, and residual components from custom GluonTS datasets
        # Compute forward pass to get Student's t-distribution arguments
        distr_args, local_loss = self(x=past_target)

        # Create Student's t-distribution
        student_t_distr = self.distr_output.distribution(distr_args)

        # TODO: once you get a prototype working, change the code to compute different losses based on output distribution
        # Compute Student's t negative log-likelihood loss
        loss = -student_t_distr.log_prob(future_target)

        return loss.mean()

    # TODO:
    def validation_step(self, batch, batch_index):
        """
        Defines the logic for a single validation loop iteration.
        """
        pass

    # TODO:
    def test_step(self, batch, batch_index):
        """
        Defines the logic for a single test loop iteration.
        """
        pass

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        return optimizer

    def get_predictor(
        self,
        input_transform,
    ):
        """
        Returns predictor for performing inference.
        """
        return PyTorchPredictor(
            prediction_length=prediction_length,
            input_names=["past_target"],
            prediction_net=super,
            batch_size=batch_size,
            input_transform=input_transform,
            forecast_generator=DistributionForecastGenerator(self.distr_output),
        )

Loaded backend agg version v2.2.


### Training Dataloader

Before creating the training set's dataloader, we'll do some preprocessing.

In [ ]:
from gluonts.dataset.field_names import FieldName
from gluonts.transform import (
    AddObservedValuesIndicator,
    InstanceSplitter,
    ExpectedNumInstanceSampler,
)

# Impute `nan`s in the target field with 0 and add a field indicating which
# values were imputed.
mask_unobserved = AddObservedValuesIndicator(
    target_field=FieldName.TARGET,
    output_field=FieldName.OBSERVED_VALUES,
)

# Split instances in the trianing set.
instance_sampler = ExpectedNumInstanceSampler(
    num_instances=1,
    min_future=prediction_length,
)

training_splitter = InstanceSplitter(
    target_field=FieldName.TARGET,
    is_pad_field=FieldName.IS_PAD,
    start_field=FieldName.START,
    forecast_start_field=FieldName.FORECAST_START,
    instance_sampler=instance_sampler,
    past_length=context_length,
    future_length=prediction_length,
    time_series_fields=[FieldName.OBSERVED_VALUES],
)

Once we're done with preprocesing, we'll create the training set's dataloader.

In [ ]:
from gluonts.dataset.loader import TrainDataLoader
from gluonts.torch.batchify import batchify

data_loader = TrainDataLoader(
    train_set,
    batch_size=batch_size,
    stack_fn=batchify,
    transform=mask_unobserved + training_splitter,
    num_batches_per_epoch=num_batches_per_epoch,
)

Now that we have a **PyTorch Lightning wrapper** and a **training dataloader**, we can instantiate the model and train it.

In [9]:
from omegaconf import OmegaConf
from pytorch_lightning import Trainer

# Load model configuration
configs = OmegaConf.load("./configs/run_TEMPO.yml")

# Initialize model wrapped with PyTorch Lightning
model = LightningTEMPO(configs)

# Initialize PyTorch Lightning trainer
trainer = Trainer(max_epochs=max_epochs)

# Train model
trainer.fit(model, data_loader)

------------------No need to load pretrained GPT model------------------


Starting new HTTPS connection (1): huggingface.co:443
https://huggingface.co:443 "HEAD /gpt2/resolve/main/tokenizer_config.json HTTP/11" 200 0
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/peft/tuners/lora/layer.py:1150: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/mike_gee/miniconda3/envs/tempo/lib/python3.8/s ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `te

Trainable params: 308736 || All params: 82207488


Training: |          | 0/? [00:00<?, ?it/s]

open file: /home/mike_gee/TEMPO/lightning_logs/version_85/checkpoints/epoch=0-step=50.ckpt
`Trainer.fit` stopped: `max_epochs=1` reached.
open file: /home/mike_gee/TEMPO/lightning_logs/version_85/hparams.yaml


## Inference

### Creating a GluonTS Predictor

To perform inference using a GluonTS model and dataset, we need to call the model's `get_predictor()` method to get a [`Predictor`](https://ts.gluon.ai/stable/api/gluonts/gluonts.torch.model.predictor.html?highlight=predictor#module-gluonts.torch.model.predictor). First, we'll split the instances in the test set

In [ ]:
from gluonts.transform import TestSplitSampler

prediction_splitter = InstanceSplitter(
    target_field=FieldName.TARGET,
    is_pad_field=FieldName.IS_PAD,
    start_field=FieldName.START,
    forecast_start_field=FieldName.FORECAST_START,
    instance_sampler=TestSplitSampler(),
    past_length=context_length,
    future_length=prediction_length,
    time_series_fields=[FieldName.OBSERVED_VALUES],
)

Then, we'll get a predictor from our model and use it to compute forecasts

In [ ]:
predictor = model.get_predictor(mask_unobserved + prediction_splitter)

ValidationError: 1 validation error for PyTorchPredictorModel
prediction_net
  instance of Module expected (type=type_error.arbitrary_type; expected_arbitrary_type=Module)

### Model Evaluation

For example, we can do backtesting on the test dataset: in what follows, `make_evaluation_predictions` will slice out the trailing `prediction_length` observations from the test time series, and use the given predictor to obtain forecasts for the same time range.

In [ ]:
from gluonts.evaluation import make_evaluation_predictions, Evaluator

In [ ]:
forecast_it, ts_it = make_evaluation_predictions(
    dataset=dataset.test, predictor=predictor_pytorch
)

forecasts_pytorch = list(f.to_sample_forecast() for f in forecast_it)
tss_pytorch = list(ts_it)

Once we have the forecasts, we can plot them:

In [ ]:
plt.figure(figsize=(20, 15))
date_formater = mdates.DateFormatter("%b, %d")
plt.rcParams.update({"font.size": 15})

for idx, (forecast, ts) in islice(enumerate(zip(forecasts_pytorch, tss_pytorch)), 9):
    ax = plt.subplot(3, 3, idx + 1)
    plt.plot(ts[-5 * prediction_length :].to_timestamp(), label="target")
    forecast.plot()
    plt.xticks(rotation=60)
    ax.xaxis.set_major_formatter(date_formater)

plt.gcf().tight_layout()
plt.legend()
plt.show()

And we can compute evaluation metrics, that summarize the performance of the model on our test data.

In [ ]:
evaluator = Evaluator(quantiles=[0.1, 0.5, 0.9])

In [ ]:
metrics_pytorch, _ = evaluator(tss_pytorch, forecasts_pytorch)
pd.DataFrame.from_records(metrics_pytorch, index=["FeedForward"]).transpose()